**CONTEXT MANAGEMENT - Context management in CrewAI is how you give agents the information they need, such as earlier task outputs, shared knowledge, or remembered details, so they can complete their tasks.**

| Mechanism                | Best used for                                             |
| ------------------------ | --------------------------------------------------------- |
| Sequential task output   | Passing the previous task’s work to the next task         |
| `context=[...]`          | Selecting particular earlier task outputs                 |
| Crew `knowledge_sources` | Giving agents common reference information                |
| `memory=True`            | Recalling relevant information across tasks and crew runs |


**1. Passing output from one task to another**

In [1]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="gpt-4o-mini", temperature=0)

researcher = Agent(
    role="Researcher",
    goal="Find simple facts about a topic",
    backstory="You explain topics clearly.",
    llm=llm
)

writer = Agent(
    role="Writer",
    goal="Turn research into a short explanation",
    backstory="You write for beginners.",
    llm=llm
)

research_task = Task(
    description="List two uses of AI in hospitals.",
    expected_output="Two short points.",
    agent=researcher
)

writing_task = Task(
    description="Explain the researcher's points in one simple sentence.",
    expected_output="One beginner-friendly sentence.",
    agent=writer
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential
)

result = await crew.kickoff_async()
print(result.raw)

AI helps doctors by analyzing medical images to find issues like tumors or fractures faster and more accurately, and it also uses chatbots to assess patients' symptoms and direct them to the right care.


**research_task produced the points; writing_task used them to write the final answer.**

**2. TASK CONTEXT - Use context=[...] when a task should receive the outputs of specific tasks. Here, the writer gets both the benefits and risks**

In [2]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="gpt-4o-mini", temperature=0)

benefits_agent = Agent(
    role="Benefits Researcher",
    goal="Identify benefits",
    backstory="You find practical benefits.",
    llm=llm
)

risks_agent = Agent(
    role="Risks Researcher",
    goal="Identify risks",
    backstory="You find practical risks.",
    llm=llm
)

writer = Agent(
    role="Summary Writer",
    goal="Write balanced summaries",
    backstory="You explain both sides simply.",
    llm=llm
)

benefits_task = Task(
    description="Give one benefit of AI in education.",
    expected_output="One benefit.",
    agent=benefits_agent
)

risks_task = Task(
    description="Give one risk of AI in education.",
    expected_output="One risk.",
    agent=risks_agent
)

summary_task = Task(
    description="Using the provided task context, summarize the benefit and risk.",
    expected_output="Two short sentences.",
    agent=writer,
    context=[benefits_task, risks_task]
)

crew = Crew(
    agents=[benefits_agent, risks_agent, writer],
    tasks=[benefits_task, risks_task, summary_task],
    process=Process.sequential
)

result = await crew.kickoff_async()
print(result.raw)

One benefit of AI in education is personalized learning, which allows for tailored educational experiences that meet each student's unique needs, enhancing engagement and improving academic outcomes. Conversely, a risk of AI in education is the potential for data privacy violations, as the collection and analysis of personal data could lead to misuse, unauthorized access, and ethical concerns regarding consent and data protection.


**context=[benefits_task, risks_task] explicitly supplied those two task results to the summary task.**

**3. SHARED CREW CONTEXT - A crew-level knowledge source is available to all agents in that crew.**

In [3]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

llm = LLM(model="gpt-4o-mini", temperature=0)

company_info = StringKnowledgeSource(
    content=(
        "Northstar Academy offers Python and AI courses. "
        "Classes are available online. "
        "Each course includes a completion certificate."
    )
)

course_agent = Agent(
    role="Course Assistant",
    goal="Answer questions about courses",
    backstory="You help students choose courses.",
    llm=llm
)

support_agent = Agent(
    role="Support Assistant",
    goal="Answer questions about course access",
    backstory="You help students understand how classes work.",
    llm=llm
)

course_task = Task(
    description="Which subjects does Northstar Academy teach?",
    expected_output="A short answer based on company information.",
    agent=course_agent
)

support_task = Task(
    description="Are classes online? Is a certificate included?",
    expected_output="A short answer based on company information.",
    agent=support_agent
)

crew = Crew(
    agents=[course_agent, support_agent],
    tasks=[course_task, support_task],
    process=Process.sequential,
    knowledge_sources=[company_info]
)

result = await crew.kickoff_async()

print("Course answer:", result.tasks_output[0].raw)
print("Support answer:", result.tasks_output[1].raw)

Course answer: Northstar Academy teaches Python and AI courses. Classes are available online, and each course includes a completion certificate.
Support answer: Yes, classes are online, and each course includes a completion certificate.


**Both agents could consult company_info. The second agent may also see the first task’s output because the crew runs sequentially.**

**4. MAINTAINING CONTEXT BETWEEN CONVERSATIONS - Set memory=True when the crew should store and recall relevant information across runs.**

In [4]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="gpt-4o-mini", temperature=0)

assistant = Agent(
    role="Learning Assistant",
    goal="Help the student using their stated preferences",
    backstory="You give clear, beginner-friendly explanations.",
    llm=llm
)

reply_task = Task(
    description="Respond briefly to the student's message: {message}",
    expected_output="A short, helpful reply.",
    agent=assistant
)

crew = Crew(
    agents=[assistant],
    tasks=[reply_task],
    process=Process.sequential,
    memory=True
)

first = await crew.kickoff_async(inputs={"message": "I prefer simple Python examples."})
print("First reply:", first.raw)
second = await crew.kickoff_async(inputs={"message": "How should you explain loops to me?"})
print("Second reply:", second.raw)

First reply: Got it! I'll make sure to provide simple Python examples for you. If you have a specific topic in mind, feel free to let me know!
Second reply: To explain loops, I would use simple Python examples. A loop allows you to repeat a block of code multiple times. For instance, a `for` loop can be used to print numbers from 1 to 5 like this:

```python
for i in range(1, 6):
    print(i)
```

This code will print each number from 1 to 5. Would you like to see more examples or a different type of loop?


**The first run gave the crew a preference. Memory allows the agent to recall that preference during the second run. Memory retrieval is relevance based, so the exact wording and recalled details can vary.**